# 01 · Fuentes y preparación

**Clases 2 y 3:** ETL y calidad de datos. Datos observados: PVOD (China, potencia y sensores), CAMMESA (Argentina mensual), Open-Meteo y NASA POWER (contexto meteorológico).

Este notebook ejecuta el constructor real. Las fuentes ya descargadas se reutilizan. Para descargar desde cero se necesita internet; para la defensa se conservan los procesados y el dashboard local. Las carpetas se definen en `src/config.py`.

In [ ]:
from pathlib import Path
import os, sys
if 'google.colab' in sys.modules:
    import subprocess
    root = Path('/content/tpo-solar')
    if not root.exists():
        subprocess.run(['git','clone','https://github.com/imelinc/tpo-solar.git',str(root)],check=True)
    os.chdir(root)
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
from src.config import DATA_RAW, DATA_PROCESSED
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go


In [ ]:
import subprocess
esperados = ['dataset_A_pvod_hourly.parquet', 'dataset_B_argentina_mensual.parquet']
if not all((DATA_PROCESSED / f).exists() for f in esperados):
    env = {**os.environ, 'TPO_DIR': str(DATA_PROCESSED), 'TPO_RAW': str(DATA_RAW)}
    subprocess.run([sys.executable, '-m', 'src.build_datasets'], cwd=ROOT, env=env, check=True)
else:
    print('Procesados disponibles: se reutiliza la copia descargada.')
for nombre in esperados:
    d = pd.read_parquet(DATA_PROCESSED / nombre)
    print(nombre, d.shape, f'{(DATA_PROCESSED / nombre).stat().st_size / 1e6:.2f} MB')


## Verificar la calidad antes de modelar

No se reemplaza un dato ausente por cero. Un cero medido puede ser una parada. La capacidad no confiable y los sensores se marcan para no interpretar su ausencia como funcionamiento normal.

In [ ]:
A = pd.read_parquet(DATA_PROCESSED / esperados[0])
B = pd.read_parquet(DATA_PROCESSED / esperados[1])
print('Duplicados China:', A.duplicated(['plant_id','timestamp_utc']).sum())
print('Duplicados Argentina:', B.duplicated(['central','fecha']).sum())
print('Faltantes por columna de Argentina:')
print(B.isna().mean().sort_values(ascending=False).head(12))
assert A.duplicated(['plant_id','timestamp_utc']).sum() == 0
assert B.duplicated(['central','fecha']).sum() == 0


## Fuentes

- PVOD: doi:10.11922/sciencedb.01094; Yao et al., Solar Energy 230 (2021).
- CAMMESA: bases Energías Renovables 2026-09 y Generación Renovable Variable 2025-09.
- Weather data by Open-Meteo.com; NASA POWER.
- El diccionario de datos documenta unidades, unión temporal y precisión de las coordenadas.
